# Hallucination Detection Pipeline – 5K WildChat Conversations (Kaggle)

This notebook runs the response-generation pipeline on **5,000 pre-selected**
WildChat conversations. Designed to run on **Kaggle** with GPU.

**Steps:**
1. Clone the GitLab repo & install dependencies
2. Load conversation hashes from `5000_convhash_labels.jsonl`
3. Load WildChat dataset & filter to target hashes
4. Load Qwen3-8B, generate responses, capture all-layer hidden states
5. Save `.pt` files + Parquet metadata

**Resume support:** already-saved `.pt` files are auto-detected and skipped.

## 0. Clone Repo & Install Dependencies

Update the `GITLAB_REPO_URL` below with your actual GitLab repo URL.
If the repo is private, use a personal access token:
`https://<username>:<token>@gitlab.com/your/repo.git`

In [ ]:
import os

# ── EDIT THIS: your GitLab repo URL ──
GITLAB_REPO_URL = "https://gitlab.com/your-username/halluc_detect.git"  # <-- CHANGE THIS

WORK_DIR = "/kaggle/working"
REPO_DIR = os.path.join(WORK_DIR, "halluc_detect")

if not os.path.exists(REPO_DIR):
    !git clone {GITLAB_REPO_URL} {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR}")
else:
    !cd {REPO_DIR} && git pull
    print(f"Repo already exists, pulled latest changes.")

# Install dependencies
!pip install -q torch transformers datasets accelerate pandas pyarrow tqdm sentencepiece

## 1. Setup & Imports

In [ ]:
import json
import logging
import re
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Optional

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Add the response_gen module to the path
RESPONSE_GEN_DIR = Path(REPO_DIR) / "response_gen"
if str(RESPONSE_GEN_DIR) not in sys.path:
    sys.path.insert(0, str(RESPONSE_GEN_DIR))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("pipeline_notebook")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Configuration

Edit paths, model parameters, or generation settings below.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
JSONL_PATH = Path(REPO_DIR) / "response_gen" / "5000_convhash_labels.jsonl"
OUTPUT_DIR = Path("/kaggle/working/output")
HIDDEN_STATES_DIR = OUTPUT_DIR / "hidden_states"
PARQUET_PATH = OUTPUT_DIR / "results.parquet"

# ── Model ──────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3-8B"
TORCH_DTYPE = "bfloat16"          # model loading dtype
SAVE_DTYPE = "float16"             # dtype for saved hidden-state tensors
DEVICE_MAP = "auto"
ENABLE_THINKING = False

# ── Generation ─────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 500
TEMPERATURE = 0.7
TOP_P = 0.9
DO_SAMPLE = True

# ── Dataset ────────────────────────────────────────────────────────────
DATASET_NAME = "allenai/WildChat-4.8M-Full"
DATASET_SPLIT = "train"

# ── Misc ───────────────────────────────────────────────────────────────
SEED = 42
LOG_EVERY = 50  # log progress every N samples

# ── Derived dtypes ─────────────────────────────────────────────────────
DTYPE_MAP = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}
model_torch_dtype = DTYPE_MAP[TORCH_DTYPE]
save_torch_dtype = DTYPE_MAP[SAVE_DTYPE]

# Create output dirs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HIDDEN_STATES_DIR.mkdir(parents=True, exist_ok=True)

print(f"JSONL path       : {JSONL_PATH}  (exists: {JSONL_PATH.exists()})")
print(f"Output dir       : {OUTPUT_DIR}")
print(f"Hidden states dir: {HIDDEN_STATES_DIR}")
print(f"Model            : {MODEL_NAME}")
print(f"Model dtype      : {TORCH_DTYPE}")
print(f"Save dtype       : {SAVE_DTYPE}")

## 3. Load Conversation Hashes from JSONL

In [ ]:
def load_target_hashes(jsonl_path: Path) -> set:
    """Read conversation hashes from the JSONL file."""
    hashes = set()
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            hashes.add(record["conversation_hash"])
    return hashes


target_hashes = load_target_hashes(JSONL_PATH)
print(f"Loaded {len(target_hashes)} target conversation hashes")

## 4. Load WildChat & Filter to Target Hashes

In [ ]:
# Category regex patterns (same as data_loader.py)
CATEGORY_PATTERNS = {
    "url": [
        r"\burl\b", r"\blink(s)?\b", r"\bwebsite\b", r"\bwebpage\b",
        r"\bhomepage\b", r"\bweb\s*address\b",
        r"\bgive\s+me\s+(a|the)\s+link\b",
    ],
    "citation": [
        r"\bcit(e|ation|ations)\b", r"\breference(s)?\b",
        r"\bsources?\b", r"\bbibliograph",
        r"\bpapers?\b.*\brecommend", r"\bacademic\b.*\breference\b",
    ],
    "coding": [
        r"\bcode\b", r"\bprogram(ming)?\b", r"\bfunction\b", r"\bscript\b",
        r"\bimplement\b", r"\balgorithm\b", r"\bpython\b", r"\bjavascript\b",
        r"\bjava\b", r"\bc\+\+\b", r"\bhtml\b", r"\bcss\b", r"\bsql\b",
        r"\bapi\b", r"\bdebug\b", r"\bclass\b.*\bmethod\b",
        r"\bwrite\b.*\b(a|the)\b.*\b(function|script|program)\b",
    ],
    "factual": [
        r"^(who|what|when|where|which|how\s+many|how\s+much|how\s+old|how\s+far|how\s+long)\b",
        r"\bis\s+it\s+true\b", r"\bcapital\s+of\b", r"\bpopulation\b",
        r"\bfounded\b", r"\binvented\b", r"\bdiscovered\b",
        r"\bhistory\s+of\b", r"\btell\s+me\s+(about|a\s+fact)\b",
    ],
}

ALL_CATEGORIES = list(CATEGORY_PATTERNS.keys())


def classify_query(text: str) -> str:
    """Return the first matching category, or 'unknown'."""
    text_lower = text.lower()
    for cat in ALL_CATEGORIES:
        for pat in CATEGORY_PATTERNS[cat]:
            if re.search(pat, text_lower):
                return cat
    return "unknown"

In [ ]:
logger.info("Loading dataset %s (split=%s) ...", DATASET_NAME, DATASET_SPLIT)
ds = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
logger.info("Dataset loaded: %d rows", len(ds))

In [ ]:
def extract_queries_by_hash(ds, target_hashes: set) -> tuple:
    """Iterate through the dataset and extract queries for matching hashes."""
    queries = []
    found_hashes = set()

    for row in tqdm(ds, desc="Scanning WildChat for target hashes"):
        conv_hash = row.get("conversation_hash", "")
        if conv_hash not in target_hashes or conv_hash in found_hashes:
            continue

        # Extract the first user message
        conversation = row.get("conversation", [])
        if not conversation:
            continue

        first_user_msg = None
        for msg in conversation:
            if msg.get("role") == "user":
                first_user_msg = msg["content"]
                break

        if not first_user_msg or len(first_user_msg.strip()) < 10:
            logger.warning("Hash %s: first user message too short or missing, skipping.", conv_hash)
            found_hashes.add(conv_hash)
            continue

        category = classify_query(first_user_msg)

        queries.append({
            "conversation_hash": conv_hash,
            "query_text": first_user_msg.strip(),
            "category": category,
        })
        found_hashes.add(conv_hash)

        # Early stop if we found all target hashes
        if len(found_hashes) >= len(target_hashes):
            break

    return queries, found_hashes


queries, found_hashes = extract_queries_by_hash(ds, target_hashes)

missing = target_hashes - found_hashes
print(f"\nMatched {len(queries)} conversations out of {len(target_hashes)} target hashes")
if missing:
    print(f"WARNING: {len(missing)} hashes not found in the dataset.")
    print(f"  First 10 missing: {list(missing)[:10]}")

# Category distribution
cat_counts = Counter(q["category"] for q in queries)
print(f"\nCategory distribution:")
for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat:12s}: {count}")

## 5. Resume Support – Detect Already-Processed Hashes

In [ ]:
def get_already_processed(hidden_states_dir: Path) -> set:
    """Return set of query_ids that already have a .pt file."""
    done = set()
    if hidden_states_dir.exists():
        for pt_file in hidden_states_dir.glob("*.pt"):
            done.add(pt_file.stem)
    return done


already_done = get_already_processed(HIDDEN_STATES_DIR)
queries_to_run = [q for q in queries if q["conversation_hash"] not in already_done]

print(f"Already processed: {len(already_done)}")
print(f"Remaining to run : {len(queries_to_run)}")

## 6. Load Model & Tokenizer

In [ ]:
logger.info("Loading tokenizer from %s ...", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

logger.info("Loading model from %s (dtype=%s) ...", MODEL_NAME, TORCH_DTYPE)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_torch_dtype,
    device_map=DEVICE_MAP,
    trust_remote_code=True,
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

logger.info("Model loaded successfully on %s", model.device)

## 7. Helper Functions

In [ ]:
def prepare_input(query: str):
    """Format the query with the chat template and tokenize."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": query},
    ]
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    encoded = tokenizer(text, return_tensors="pt", padding=False, truncation=True)
    device = model.device
    return encoded.input_ids.to(device), encoded.attention_mask.to(device)


@torch.no_grad()
def generate_with_hidden_states(query: str):
    """
    Two-pass generation:
      Pass 1: fast autoregressive generation
      Pass 2: single forward pass to capture all-layer hidden states

    Returns: (response_text, generated_token_ids, query_hidden_states, response_hidden_states)
    """
    input_ids, attention_mask = prepare_input(query)
    prompt_len = input_ids.shape[1]

    # Pass 1: generate
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )
    full_seq = outputs[0]  # (total_len,)
    gen_ids = full_seq[prompt_len:]
    response_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    # Pass 2: forward pass for hidden states
    fwd_out = model(
        input_ids=full_seq.unsqueeze(0),
        output_hidden_states=True,
    )

    save_kwargs = dict(dtype=save_torch_dtype, device="cpu")

    # Query (prompt) hidden states: (num_prompt_tokens, num_layers+1, hidden_dim)
    query_layers = torch.stack(
        [layer[0, :prompt_len, :] for layer in fwd_out.hidden_states]
    )
    query_hs = query_layers.transpose(0, 1).contiguous().to(**save_kwargs)

    # Response hidden states: (num_gen_tokens, num_layers+1, hidden_dim)
    response_layers = torch.stack(
        [layer[0, prompt_len:, :] for layer in fwd_out.hidden_states]
    )
    response_hs = response_layers.transpose(0, 1).contiguous().to(**save_kwargs)

    return response_text, gen_ids.cpu(), query_hs, response_hs


def save_sample(
    query_id, query_text, category, response_text,
    generated_token_ids, hidden_states, query_hidden_states,
):
    """Save hidden states to .pt file and return a metadata record."""
    pt_path = HIDDEN_STATES_DIR / f"{query_id}.pt"
    torch.save(
        {
            "query_id": query_id,
            "hidden_states": hidden_states,
            "query_hidden_states": query_hidden_states,
            "generated_token_ids": generated_token_ids,
        },
        pt_path,
    )
    return {
        "query_id": query_id,
        "query_text": query_text,
        "category": category,
        "response_text": response_text,
        "num_generated_tokens": len(generated_token_ids),
        "num_prompt_tokens": query_hidden_states.shape[0],
        "hidden_states_path": str(pt_path),
    }

## 8. Run the Pipeline

In [ ]:
records = []
num_errors = 0

logger.info("Starting generation for %d queries ...", len(queries_to_run))
start_time = time.time()

for idx, q in enumerate(tqdm(queries_to_run, desc="Generating")):
    query_id = q["conversation_hash"]

    try:
        response_text, gen_ids, query_hs, response_hs = generate_with_hidden_states(
            q["query_text"]
        )
    except Exception as e:
        logger.exception("Failed on query %s – skipping.", query_id)
        num_errors += 1
        continue

    record = save_sample(
        query_id=query_id,
        query_text=q["query_text"],
        category=q["category"],
        response_text=response_text,
        generated_token_ids=gen_ids,
        hidden_states=response_hs,
        query_hidden_states=query_hs,
    )
    records.append(record)

    # Periodic logging
    if (idx + 1) % LOG_EVERY == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / (elapsed / 60)
        logger.info(
            "Progress: %d / %d  (%.1f samples/min, %d errors so far)",
            idx + 1, len(queries_to_run), rate, num_errors,
        )

elapsed = time.time() - start_time
logger.info(
    "Done: %d samples in %.1f min (%.2f s/sample), %d errors.",
    len(records), elapsed / 60, elapsed / max(len(records), 1), num_errors,
)

## 9. Save Metadata to Parquet

In [ ]:
if records:
    df_new = pd.DataFrame(records)

    # If a previous parquet exists (from a resumed run), merge with it
    if PARQUET_PATH.exists():
        df_old = pd.read_parquet(PARQUET_PATH)
        df_combined = pd.concat([df_old, df_new], ignore_index=True)
        df_combined = df_combined.drop_duplicates(subset="query_id", keep="last")
        df_combined.to_parquet(PARQUET_PATH, index=False)
        print(f"Merged with existing parquet. Total rows: {len(df_combined)}")
    else:
        df_new.to_parquet(PARQUET_PATH, index=False)
        print(f"Saved metadata: {len(df_new)} rows")

    print(f"Parquet saved to: {PARQUET_PATH}")
else:
    print("No new records to save.")

## 10. Summary

In [ ]:
total_pt_files = len(list(HIDDEN_STATES_DIR.glob("*.pt")))

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Target hashes         : {len(target_hashes)}")
print(f"Found in WildChat     : {len(queries)}")
print(f"Previously processed  : {len(already_done)}")
print(f"Processed this run    : {len(records)}")
print(f"Errors this run       : {num_errors}")
print(f"Total .pt files saved : {total_pt_files}")
print(f"Parquet path          : {PARQUET_PATH}")
print("=" * 60)

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    print(f"\nParquet preview ({len(df)} rows):")
    display(df.head())
    print(f"\nCategory distribution in parquet:")
    display(df["category"].value_counts())